In [0]:
from pyspark.sql.functions import col, mean, stddev, abs, monotonically_increasing_id

df = (
    spark.read.format("delta").load("dbfs:/user/hive/warehouse/sample_anomaly_data")
)

numeric_col = "value"

dfIndexed = df.withColumn("index", monotonically_increasing_id() + 1)

stats = dfIndexed.select(
    mean(col(numeric_col)).alias("mean"),
    stddev(col(numeric_col)).alias("stddev")
).collect()[0]
mean_val = stats["mean"]
stddev_val = stats["stddev"]

outliers_df = dfIndexed.withColumn(
    "z_score",
    (col(numeric_col) - mean_val) / stddev_val
).filter(abs(col("z_score")) > 2)

df_woOutliers = dfIndexed.join(
    outliers_df.select("index"),
    on="index",
    how="left_anti"
)

spark.sql("""
    CREATE TABLE IF NOT EXISTS values_compared (
        goodVal INT,
        anomVal INT
    )
    USING DELTA
""")

spark.sql("""
            TRUNCATE TABLE hive_metastore.default.values_compared""")


spark.sql("""
    DELETE FROM hive_metastore.default.sample_anomaly_data
    WHERE value >= 101 AND value <= 110
""")

df_woOutliers.createOrReplaceGlobalTempView("goodValues")
outliers_df.createOrReplaceGlobalTempView("anomalies")

spark.sql("""
    INSERT INTO hive_metastore.default.values_compared (goodVal, anomVal)
    SELECT value, NULL FROM global_temp.goodValues
""")

spark.sql("""
    INSERT INTO hive_metastore.default.values_compared (goodVal, anomVal)
    SELECT NULL, value FROM global_temp.anomalies
""")

df = spark.table("hive_metastore.default.values_compared")
display(df)

In [0]:
mea